In [ ]:
# ==============================================================================
# 1. SETUP & SSH AUTHENTICATION FOR GOOGLE COLAB
# Prerequisite: Upload both 'garmin_combined_summaries_from_2026-11-14.csv' 
# and your private SSH key ('sas_key') into Colab's main /content folder.
# ==============================================================================

# Clear unused R memory
gc()

# Set working directory to Colab's default root folder
setwd("/content")

# Create the standard .ssh folder and move your uploaded key into it
system("mkdir -p ~/.ssh")
system("cp /content/sas_key ~/.ssh/sas_key")

# Set strict permissions on the private key (SSH will reject it otherwise)
system("chmod 600 ~/.ssh/sas_key")

# Add GitHub to known_hosts so SSH doesn't hang waiting for a 'yes/no' prompt
system("ssh-keyscan github.com >> ~/.ssh/known_hosts")

# Configure Git to automatically use your specific SSH key and set your identity
Sys.setenv(GIT_SSH_COMMAND = "ssh -i ~/.ssh/sas_key -o IdentitiesOnly=yes")
system('git config --global user.name "mahoneyjustinnj"')
system('git config --global user.email "mahoneyjustinnj@gmail.com"')

# Test SSH authentication with GitHub (should return: "Hi mahoneyjustinnj!...")
system("ssh -T git@github.com")


# ==============================================================================
# 2. CLONE REPO & MOVE YOUR CSV FILE INTO IT
# ==============================================================================

# Clone your GitHub repository into Colab (skips if already cloned in this session)
if (!dir.exists("/content/myGarminHealth_JM")) {
  system("git clone git@github.com:mahoneyjustinnj/myGarminHealth_JM.git")
}

# Move into the cloned repository folder
setwd("/content/myGarminHealth_JM")

# Pull the latest changes from GitHub to ensure the branch is up to date
system("git pull origin main")

# Copy your uploaded CSV from /content into the repository folder
file.copy(
  from = "/content/garmin_combined_summaries_from_2026-11-14.csv",
  to   = "/content/myGarminHealth_JM/garmin_combined_summaries_from_2026-11-14.csv",
  overwrite = TRUE
)


# ==============================================================================
# 3. STAGE, COMMIT, TAG, AND PUSH TO GITHUB
# ==============================================================================

# Grab today's date dynamically (e.g., "2026-11-14")
today_str <- format(Sys.Date(), "%Y-%m-%d")

# Check Git status (make sure your CSV isn't being blocked by .gitignore!)
system("git status")

# Stage ONLY your specific CSV file (use -f to force-add if .gitignore blocks .csv files)
system("git add -f garmin_combined_summaries_from_2026-11-14.csv")

# Commit the file with today's date in the message
commit_cmd <- sprintf('git commit -m "Daily update of Garmin CSVs %s"', today_str)
system(commit_cmd)

# Push the commit to the 'main' branch on GitHub
system("git push origin main")

# Create a version tag for today's update (e.g., v2026-11-14) and push it
tag_name <- paste0("v", today_str)
system(sprintf("git tag -f %s", tag_name))
system(sprintf("git push -f origin %s", tag_name))put